# `n_color_immerse()`

This document explains why assigning colors to three-dimensional nematic directors is a surprisingly nontrivial visualization problem. At first glance, a 3D director has three components and an RGB color also has three components, so one might expect a simple one-to-one coloring rule. However, the nematic symmetry $\mathbf n\sim-\mathbf n$ changes the topology of the orientation space and makes the most natural requirements on such a colormap mutually incompatible.

We will first explain where this obstruction comes from, why an ordinary one-to-one continuous coloring is mathematically impossible, and why an immersion provides the appropriate compromise.

## Background

A three-dimensional nematic director is represented by a unit vector $\mathbf n\in S^2$, but the two vectors $\mathbf n$ and $-\mathbf n$ describe the same physical orientation. The space of distinct nematic orientations is therefore not the sphere itself, but the real projective plane

$$
\mathbb{RP}^2=S^2/(\mathbf n\sim-\mathbf n).
$$

A director colormap can thus be viewed as a map from $\mathbb{RP}^2$ into RGB color space. If we ignore the finite bounds of RGB for the moment, color space is locally just three-dimensional Euclidean space $\mathbb R^3$. The most desirable colormap would be continuous and one-to-one: nearby orientations would have nearby colors, while every distinct orientation would have a distinct color. Topologically, this would amount to an **embedding** of $\mathbb{RP}^2$ in $\mathbb R^3$.

Such an embedding does not exist. The real projective plane is a closed non-orientable surface, and it cannot be embedded in three-dimensional Euclidean space without self-intersection. Consequently, no continuous RGB colormap can simultaneously preserve the nematic identification and assign a globally unique color to every director orientation. This is a topological obstruction rather than a limitation of a particular coloring formula.

The obstruction becomes much less restrictive if we replace an embedding by an **immersion**. An immersion is locally regular: around every point, the surface is mapped smoothly into $\mathbb R^3$ without collapsing a local direction. Unlike an embedding, however, an immersion is allowed to intersect itself globally. Thus two distinct points of $\mathbb{RP}^2$ may occasionally be mapped to the same point in $\mathbb R^3$, while the map still preserves the local two-dimensional structure of orientation space.

This distinction is exactly what we need for director coloring. Since a perfect one-to-one continuous map is impossible, we instead seek an immersion whose self-intersections are as controlled as possible and whose geometry makes different orientations as distinguishable as possible after conversion to RGB colors.

A classical realization of this idea is **Boy's surface**, an immersion of $\mathbb{RP}^2$ into $\mathbb R^3$. Boy's surface provides a concrete geometric model of the nematic orientation space in three dimensions: the unavoidable topological conflict is concentrated into self-intersections rather than resolved by introducing a discontinuity or an arbitrary sign convention for $\mathbf n$. This makes it a natural starting point for constructing a three-dimensional nematic colormap.

There is not, however, a unique Boy's surface. Different immersions and different geometric realizations distribute distortion and self-intersection differently. For visualization, these differences matter: after the immersed coordinates are transformed into a realizable color space, they determine how well separated the resulting colors are and how uniformly orientation changes are represented. The next problem is therefore not merely to find *a* Boy's surface, but to find a realization that is particularly suitable for coloring nematic directors.

## Design goals

Our goal is now to find an **optimal Boy's surface** for director coloring. Rather than searching over arbitrary immersions of $\mathbb{RP}^2$, we start from a fixed Boy's-surface immersion and apply an invertible affine transformation in $\mathbb R^3$. If the original immersion is written as

$$
\mathbf f(\mathbf n):\mathbb{RP}^2\rightarrow\mathbb R^3,
$$

we consider the family

$$
\mathbf f_{A,\mathbf b}(\mathbf n)=A\,\mathbf f(\mathbf n)+\mathbf b,
$$

where $A$ is an invertible $3\times3$ matrix and $\mathbf b$ is a translation vector. Because an invertible affine map is a diffeomorphism of $\mathbb R^3$, this transformation does not change the topology of the immersed projective plane: it preserves the nematic identification, does not introduce singularities into the immersion, and does not alter the basic topological structure of its self-intersections. It only changes the geometry and placement of the realization in three-dimensional space.

This gives us a finite-dimensional optimization problem. The entries of $A$ together with the components of $\mathbf b$ are treated as adjustable parameters, and we search for the affine transformation that maximizes a utility function measuring how suitable the resulting surface is for visualization. In other words, instead of asking which Boy's surface is mathematically preferable in the abstract, we ask which affine deformation of a valid Boy's surface best serves the practical purpose of distinguishing nematic orientations by color.

The remaining question is therefore how to define *best*. Several properties matter for a useful director colormap, and they need not favor the same transformation. We introduce these requirements separately below.

### Locally uniform color variation

The first requirement is **local uniformity**. Ideally, the perceptual difference between nearby colors should change in proportion to the corresponding local change of director orientation. In other words, the colormap should distort the local geometry of orientation space as little as possible: a small director change should not look dramatic in one region of the map but almost invisible in another.

The optimization measures this requirement through the **local metric** rather than through a finite-distance formula between two arbitrary directors. A director is a unit vector on $\mathbb S^2$, so its infinitesimal variations lie in the two-dimensional tangent plane. We use the ordinary Euclidean metric inherited from $\mathbb R^3$ on this tangent plane. Equivalently, if $(\mathbf e_1,\mathbf e_2)$ is an orthonormal tangent basis and $\delta\mathbf n=u_1\mathbf e_1+u_2\mathbf e_2$, then

$$
d\ell_{\mathrm{director}}^2=u_1^2+u_2^2.
$$

This is the natural infinitesimal angular metric of the unit sphere. The nematic identification $\mathbf n\sim-\mathbf n$ does not alter this local metric.

For colors, ordinary Euclidean distance in RGB coordinates is not appropriate. RGB is designed to encode display intensities, not human perceptual similarity: equal numerical changes in RGB can look very different depending on where they occur in color space. We instead use **OKLab**, a perceptual color space designed so that Euclidean distances approximately track perceived color differences. If the color assigned to $\mathbf n$ has OKLab coordinates

$$
\mathbf q(\mathbf n)=(L(\mathbf n),a(\mathbf n),b(\mathbf n))
$$

then Euclidean distance in $\mathbf q$ provides the perceptual metric used by the optimization. Restricting the differential of the complete map $f(\mathbf n)=\operatorname{OKLab}[\mathbf c_{\rm sRGB}(\mathbf n)]$ to the director tangent plane gives the $3\times2$ matrix

$$
D\mathbf q=\left(\partial_{\mathbf e_1}\mathbf q,\partial_{\mathbf e_2}\mathbf q\right),
$$

and the induced perceptual metric is

$$
G(\mathbf n)=(D\mathbf q)^T D\mathbf q.
$$

If the color difference were exactly proportional to the director difference everywhere, $G$ would be proportional to the identity with the same proportionality scale throughout orientation space. The actual optimization measures departure from this ideal with the normalized functional

$$
J_{\mathrm{loc}}=\frac{\left\langle\operatorname{tr}\left(G^2\right)\right\rangle}{\left\langle\operatorname{tr}G\right\rangle^2}-\frac12.
$$

Here $\langle\cdot\rangle$ denotes an average over director space. The denominator normalizes out the overall size of the color surface, so the score measures geometric nonuniformity rather than rewarding a trivial global rescaling. The subtraction of $1/2$ makes the ideal two-dimensional isotropic case $G\propto I$ give $J_{\mathrm{loc}}=0$. Therefore smaller $J_{\mathrm{loc}}$ means better local metric fidelity. In the final optimization, this quantity is imposed as the constraint $J_{\mathrm{loc}}\le t$; the selected production map uses $t=0.55$.

### Recognizable Cartesian axes

For practical visualization, the three Cartesian director axes should also have immediately recognizable colors. We associate the $x$, $y$, and $z$ directors with red, green, and blue, respectively. This is not required by the topology, but it makes plots substantially easier to read: a user can identify the dominant orientation without first learning an arbitrary color convention. Let $\mathbf q_x,\mathbf q_y,\mathbf q_z$ be the OKLab colors produced for the three Cartesian directors, and let $\mathbf q_R,\mathbf q_G,\mathbf q_B$ be the OKLab coordinates of the target sRGB primaries. We define the three axis errors as

$$
d_x=\|\mathbf q_x-\mathbf q_R\|_2,\qquad d_y=\|\mathbf q_y-\mathbf q_G\|_2,\qquad d_z=\|\mathbf q_z-\mathbf q_B\|_2.
$$

A convenient single diagnostic is the RMS axis error

$$
J_{\mathrm{axis}}=\sqrt{\frac{d_x^2+d_y^2+d_z^2}{3}},
$$

for which smaller values are better. In the actual optimization, however, the three distances are constrained individually,

$$
d_x\le\delta_x,\qquad d_y\le\delta_y,\qquad d_z\le\delta_z,
$$

rather than relying only on their average. This prevents a very poor match on one axis from being hidden by excellent matches on the other two. The tolerances $\delta_x,\delta_y,\delta_z$ are calibrated from the original Nematics3D colormap, so the optimized map is not allowed to gain vividness by sacrificing these familiar reference colors.

### Vivid colors

After local uniformity and the Cartesian reference colors have been controlled, we would like the remaining map to be as vivid as possible. A mathematically valid immersion could occupy only a small, nearly gray region of color space, but such a map would make different orientations unnecessarily difficult to distinguish. In OKLab, the chroma of a color $\mathbf q=(L,a,b)$ is

$$
C(\mathbf n)=\sqrt{a(\mathbf n)^2+b(\mathbf n)^2}.
$$

We use the mean chroma over director space as the vividness utility,

$$
U_{\mathrm{chroma}}=\left\langle C(\mathbf n)\right\rangle=\left\langle\sqrt{a(\mathbf n)^2+b(\mathbf n)^2}\right\rangle,
$$

and **larger is better**. This is the quantity that is ultimately maximized once the acceptable local-distortion and axis-color bounds have been specified. Increasing $U_{\mathrm{chroma}}$ moves the image away from the gray axis in a perceptually meaningful color space and therefore makes the available orientation colors more vivid.

There is a genuine tradeoff between vividness and local uniformity: allowing more local distortion generally permits a more colorful map. We therefore treat these quantities through a Pareto analysis rather than pretending that one arbitrary weighted sum defines the answer. The selected map is taken from the resulting chroma--$J_{\mathrm{loc}}$ Pareto frontier while respecting the axis-color requirements.

### The sRGB gamut: a hard constraint

Finally, every color produced by the map must actually be displayable. The affine transformation acts in three-dimensional coordinates that are ultimately interpreted as encoded sRGB values, so the complete immersed surface must remain inside the sRGB cube,

$$
0\le R,G,B\le1.
$$

More explicitly, if $\mathbf c(\mathbf n)=(R(\mathbf n),G(\mathbf n),B(\mathbf n))=A\mathbf f(\mathbf n)+\mathbf b$, the requirement is

$$
\min_{\mathbf n} c_k(\mathbf n)\ge0,\qquad \max_{\mathbf n} c_k(\mathbf n)\le1,\qquad k\in\{R,G,B\}.
$$

This is fundamentally different from the utilities above. Leaving the sRGB gamut is not merely undesirable; it produces an invalid display color and would require clipping, which would deform the optimized map and can collapse distinct colors onto the gamut boundary. We therefore impose gamut containment as a **hard constraint** on the optimization rather than assigning it a penalty in the utility function.

Taken together, the optimization can be summarized schematically as

$$
\max_{A,\mathbf b} U_{\mathrm{chroma}}(A,\mathbf b)
$$

subject to an upper bound on $J_{\mathrm{loc}}$, the three calibrated axis-color bounds, and exact containment of the complete Boy's-surface image in the sRGB gamut.

## Implementation

## Usage

## Visualizing the full colormap: the color sphere